# Lab 1: Prerequisites & Infrastructure Setup

# 实验 1：先决条件和基础设施设置

## Overview

## 概述

Verify all prerequisites for the workshop and deploy the CRM application stack on AWS (EC2 + NGINX + DynamoDB).

验证研讨会的所有先决条件，并在 AWS 上部署 CRM 应用程序堆栈（EC2 + NGINX + DynamoDB）。

## Objectives

## 目标

**Part 1: Prerequisites**
- Verify Python version (3.10+)
- Verify AWS account and credentials
- Install workshop dependencies
- Verify Bedrock AgentCore SDK and starter toolkit
- Test Bedrock model access
- Set up Agent Memory for shared context
- Set up User and Agent identities using Amazon Cognito

**第一部分：先决条件**
- 验证 Python 版本（3.10+）
- 验证 AWS 账户和凭证
- 安装研讨会依赖项
- 验证 Bedrock AgentCore SDK 和入门工具包
- 测试 Bedrock 模型访问
- 设置代理记忆以共享上下文
- 使用 Amazon Cognito 设置用户和代理身份

**Part 2: Infrastructure Setup**
- Provision AWS infrastructure: EC2 instance, NGINX, DynamoDB, CloudWatch
- Deploy a sample CRM application
- Create fault injection scripts to simulate failures
- Set up CloudWatch monitoring
- Verify infrastructure is running and accessible

**第二部分：基础设施设置**
- 配置 AWS 基础设施：EC2 实例、NGINX、DynamoDB、CloudWatch
- 部署示例 CRM 应用程序
- 创建故障注入脚本以模拟故障
- 设置 CloudWatch 监控
- 验证基础设施正在运行且可访问

## What You'll Learn

## 您将学到什么

- Workshop prerequisites and setup workflow
- Fault injection patterns for testing incident response
- CloudWatch log and metric setup for diagnostics

- 研讨会先决条件和设置工作流
- 用于测试事件响应的故障注入模式
- 用于诊断的 CloudWatch 日志和指标设置

## 1. Verify Python Version

## 1. 验证 Python 版本

In [ ]:
import sys
print(f"Python version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
assert sys.version_info >= (3, 10), "Python 3.10+ required"
print("✅ Python version check passed")

## 2. Install Workshop Dependencies

## 2. 安装研讨会依赖项

In [ ]:
%pip install -q -r requirements.txt
print("✅ Workshop dependencies installed")

## 3. Verify AWS Configuration

## 3. 验证 AWS 配置

In [ ]:
import boto3
from lab_helpers.config import AWS_REGION, AWS_PROFILE, MODEL_ID, WORKSHOP_NAME
from lab_helpers.lab_01.infrastructure import get_app_url

# Display configuration
print(f"Workshop Name: {WORKSHOP_NAME}")
print(f"AWS Region: {AWS_REGION}")
print(f"Model ID: {MODEL_ID}\n")

# Verify AWS credentials
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
sts = session.client('sts')
identity = sts.get_caller_identity()

print(f"✅ AWS Account: {identity['Account']}")
print(f"✅ AWS User/Role: {identity['Arn']}")

## 4. Test Bedrock Model Access

## 4. 测试 Bedrock 模型访问

In [ ]:
import boto3
from lab_helpers.config import AWS_REGION, MODEL_ID, AWS_PROFILE

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
bedrock = session.client('bedrock', region_name=AWS_REGION)

# Verify model access
try:
    model = bedrock.get_foundation_model(modelIdentifier=MODEL_ID)
    print(f"Model ID: {MODEL_ID}")
    print(f"✅ Bedrock model access verified")
except Exception as e:
    print(f"❌ Error accessing model: {e}")
    raise

## 5. Verify AgentCore Components

## 5. 验证 AgentCore 组件

In [ ]:
import importlib

packages = ['bedrock_agentcore', 'strands', 'boto3', 'pydantic']

for package in packages:
    try:
        mod = importlib.import_module(package)
        version = getattr(mod, '__version__', 'installed')
        print(f"✅ {package:<20} {version}")
    except ImportError:
        print(f"❌ {package:<20} NOT FOUND")

print("\n✅ All core packages verified")

## Summary

## 总结

✅ All prerequisites verified. Ready to proceed to Lab 1: Infrastructure Setup & Fault Injection.

✅ 所有先决条件已验证。准备继续进行实验 1：基础设施设置和故障注入。

## Part 1.5: Cognito Setup (Authentication for Labs 3-5)

## 第 1.5 部分：Cognito 设置（实验 3-5 的认证）

### Overview

### 概述

In this section, we'll set up AWS Cognito for authentication infrastructure used by Labs 3-5:

在本节中，我们将为实验 3-5 使用的认证基础设施设置 AWS Cognito：

**What We'll Create:**
- Cognito User Pool: `aiml301-UserPool`
- **Two User Groups** (NEW):
  - **developers**: Users who create remediation plans
  - **approvers**: Users who approve and execute plans
- **Two App Clients**:
  - **User Auth Client** (public): For end-user authentication with OAuth support
  - **M2M Client** (confidential): For Gateway-to-Runtime service-to-service authentication
- **Resource Server**: Custom scopes for fine-grained authorization (`mcp.invoke`, `runtime.access`)
- **User Pool Domain**: OAuth2 token endpoint
- **Two Test Users**:
  - **Developer User**: `testuser@aiml301.example.com` (member of `developers` group)
  - **Approver User**: `approver@aiml301.example.com` (member of `approvers` group)

**我们将创建的内容：**
- Cognito 用户池：`aiml301-UserPool`
- **两个用户组**（新增）：
  - **developers**：创建修复计划的用户
  - **approvers**：批准和执行计划的用户
- **两个应用程序客户端**：
  - **用户认证客户端**（公共）：用于支持 OAuth 的最终用户认证
  - **M2M 客户端**（机密）：用于 Gateway 到 Runtime 的服务间认证
- **资源服务器**：用于细粒度授权的自定义范围（`mcp.invoke`、`runtime.access`）
- **用户池域**：OAuth2 令牌端点
- **两个测试用户**：
  - **开发者用户**：`testuser@aiml301.example.com`（`developers` 组成员）
  - **审批者用户**：`approver@aiml301.example.com`（`approvers` 组成员）

**Authentication Flows:**
1. **User Auth** (Client → Gateway): End-users authenticate with credentials, receive JWT tokens with group membership
2. **M2M Auth** (Gateway → Runtime): Gateway uses client credentials grant to get M2M tokens for Runtime access

**认证流程：**
1. **用户认证**（客户端 → Gateway）：最终用户使用凭证进行认证，接收带有组成员身份的 JWT 令牌
2. **M2M 认证**（Gateway → Runtime）：Gateway 使用客户端凭证授权获取用于 Runtime 访问的 M2M 令牌

**Multi-Actor Workflow (Lab-03):**
- **Developer** logs in → Creates remediation plan → Gets blocked (needs approval)
- **Approver** logs in → Discovers pending incidents → Reviews plan → Approves execution
- **Developer** returns → Sees approval in shared memory → Executes approved steps

**多参与者工作流（实验-03）：**
- **开发者**登录 → 创建修复计划 → 被阻止（需要批准）
- **审批者**登录 → 发现待处理事件 → 审查计划 → 批准执行
- **开发者**返回 → 在共享记忆中看到批准 → 执行已批准的步骤

**JWT Token Claims:**
After setup, JWT ID tokens will include:

**JWT 令牌声明：**
设置后，JWT ID 令牌将包含：

```json
{
  "email": "developer@aiml301.example.com",
  "cognito:username": "developer@aiml301.example.com",
  "cognito:groups": ["developers"],
  "sub": "uuid",
  "scope": "openid profile email custom-scopes"
}
```

**Why Groups?**
- **Role-based authorization**: Gateway can check if user is in `approvers` group before allowing execution
- **Incident routing**: Only notify approvers for pending incidents
- **Audit trails**: Memory records show which role performed each action
- **Actor identification**: `email` claim provides readable actor_id instead of UUID

**为什么使用组？**
- **基于角色的授权**：Gateway 可以在允许执行之前检查用户是否在 `approvers` 组中
- **事件路由**：仅通知审批者待处理的事件
- **审计跟踪**：记忆记录显示哪个角色执行了每个操作
- **参与者识别**：`email` 声明提供可读的 actor_id 而不是 UUID

### Objectives

### 目标

✅ Set up Cognito infrastructure for centralized authentication
✅ Create user groups for role-based access control
✅ Enable dual auth modes: user-based and service-to-service
✅ Create fine-grained authorization scopes
✅ Enable OAuth flows for rich JWT ID tokens
✅ Store configuration in SSM Parameter Store for use by later labs

✅ 设置 Cognito 基础设施以进行集中认证
✅ 创建用户组以进行基于角色的访问控制
✅ 启用双重认证模式：基于用户和服务间
✅ 创建细粒度授权范围
✅ 启用 OAuth 流程以获取丰富的 JWT ID 令牌
✅ 将配置存储在 SSM Parameter Store 中供后续实验使用

#### 1. Execute Cognito Setup

#### 1. 执行 Cognito 设置

In [ ]:
from lab_helpers.cognito_setup import setup_cognito_complete

# Execute complete Cognito setup workflow
cognito_config = setup_cognito_complete()

print("\n" + "="*70)
print("COGNITO SETUP COMPLETE")
print("="*70)
print("Cognito User Pool ID: ", cognito_config['user_pool_id'])

## Part 1.6: Memory Setup for Labs 2-5

## 第 1.6 部分：实验 2-5 的记忆设置

In this section, we'll create a shared AgentCore Memory resource that will be used by all agent labs (2-5) for conversation history and session management.

在本节中，我们将创建一个共享的 AgentCore Memory 资源，该资源将被所有代理实验（2-5）用于对话历史和会话管理。

### What We'll Create:

### 我们将创建的内容：

- AgentCore Memory resource with 7-day expiry
- Store memory_id in Parameter Store for easy access by Labs 2-5
- Store default session ID for static session tracking in Labs 2-4

- 具有 7 天过期时间的 AgentCore Memory 资源
- 将 memory_id 存储在 Parameter Store 中，以便实验 2-5 轻松访问
- 存储默认会话 ID，用于实验 2-4 中的静态会话跟踪

### Key Learning:

### 关键学习：

Memory enables context persistence across agent calls and multi-turn conversations. All labs will share this single memory resource.

记忆支持跨代理调用和多轮对话的上下文持久化。所有实验将共享这个单一的记忆资源。

### Objectives

### 目标

✅ Create AgentCore Memory resource  
✅ Store memory configuration in Parameter Store  
✅ Enable conversation history loading for downstream agents

✅ 创建 AgentCore Memory 资源  
✅ 将记忆配置存储在 Parameter Store 中  
✅ 为下游代理启用对话历史加载

In [ ]:
### 1.6.1: Create AgentCore Memory Resource

from bedrock_agentcore.memory import MemoryClient
from lab_helpers.constants import PARAMETER_PATHS
from datetime import datetime

memory_client = MemoryClient(region_name=AWS_REGION)
memory_name = f"{PARAMETER_PATHS['memory']['memory_name_prefix']}_{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Creating memory: {memory_name}")
memory = memory_client.create_memory_and_wait(
    name=memory_name,
    description="SRE Agent Shared Short-Term Memory for Labs 2-5",
    strategies=[],
    event_expiry_days=7,
    max_wait=600,
    poll_interval=10
)

memory_id = memory['id']
print(f"✅ Memory created: {memory_id} (Status: ACTIVE, Expiry: 7 days)")

In [ ]:
### 1.6.2: Store Memory Configuration in Parameter Store

from lab_helpers.parameter_store import put_parameter

# Store memory_id for Labs 2-5
put_parameter(
    PARAMETER_PATHS['memory']['memory_id'],
    memory_id,
    description="Memory ID for agent conversation history",
    region_name=AWS_REGION
)

# Store default session ID for Labs 2-4
put_parameter(
    PARAMETER_PATHS['memory']['default_session_id'],
    "crm-session-id",
    description="Default session ID for Labs 2-4",
    region_name=AWS_REGION
)

print(f"✅ PSM Keys stored:")
print(f"   • {PARAMETER_PATHS['memory']['memory_id']} = {memory_id}")
print(f"   • {PARAMETER_PATHS['memory']['default_session_id']} = crm-session-id")

### Summary: Memory Setup Complete

### 总结：记忆设置完成

✅ **AgentCore Memory Resource Created**
- Single shared memory resource for all labs (2-5)
- Automatic 7-day expiry for cost management
- Supports multi-turn conversations and context loading

✅ **AgentCore Memory 资源已创建**
- 所有实验（2-5）共享的单一记忆资源
- 自动 7 天过期以进行成本管理
- 支持多轮对话和上下文加载

✅ **Parameter Store Configuration**
- Memory ID stored for Lab 2-5 retrieval
- Default session ID available for Labs 2-4
- Follows central configuration pattern

✅ **Parameter Store 配置**
- 存储 Memory ID 供实验 2-5 检索
- 默认会话 ID 可用于实验 2-4
- 遵循集中配置模式

**Next Steps:**
- Lab 2: Retrieve memory_id and initialize memory hooks
- Lab 3-4: Use same memory for remediation/prevention agents
- Lab 5: Multi-agent orchestration with shared memory

**下一步：**
- 实验 2：检索 memory_id 并初始化记忆钩子
- 实验 3-4：为修复/预防代理使用相同的记忆
- 实验 5：使用共享记忆进行多代理编排

## Part 2: Infrastructure Setup & CRM Application Deployment

## 第二部分：基础设施设置和 CRM 应用程序部署

The Infrastructure Setup & CRM Application Deployment is automated is a part of the workshop set up.

基础设施设置和 CRM 应用程序部署作为研讨会设置的一部分已自动完成。

Please proceed to the next section.

请继续进行下一节。

In [ ]:
# Try url with both port 80 and 8080
print(f"Click here to access the CRM App UI: '{get_app_url()}'")


## 1. Set Up Fault Injection Utilities

## 1. 设置故障注入工具

In this section, we'll prepare tools to inject infrastructure faults and review pre-baked faults already built into the deployment. The workshop includes **4 total faults** for comprehensive SRE training:

在本节中，我们将准备工具来注入基础设施故障，并审查部署中已内置的预设故障。研讨会包括 **4 个故障**，用于全面的 SRE 培训：

- **Fault 1: DynamoDB Throttling** - Reduce table capacity to trigger ProvisionedThroughputExceededException
- **Fault 2: IAM Permission Issues** - Restrict EC2 role permissions to cause AccessDenied errors

- **故障 1：DynamoDB 限流** - 降低表容量以触发 ProvisionedThroughputExceededException
- **故障 2：IAM 权限问题** - 限制 EC2 角色权限以导致 AccessDenied 错误

These faults will be used throughout the workshop to test your SRE agent's diagnostic capabilities across different failure modes and detection methods.

这些故障将在整个研讨会中使用，以测试您的 SRE 代理在不同故障模式和检测方法下的诊断能力。

In [ ]:
from lab_helpers.lab_01.fault_injection import (
    initialize_fault_injection,
    inject_dynamodb_throttling,
    inject_iam_permissions,
)

# Initialize AWS clients and retrieve infrastructure resource IDs from SSM
print("Initializing fault injection utilities...")
resources = initialize_fault_injection(AWS_REGION, AWS_PROFILE)

print(f"\nDiscovered Infrastructure Resources:")
print(f"  Nginx Instance: {resources.get('nginx_instance_id', 'Not found')}")
print(f"  App Instance: {resources.get('app_instance_id', 'Not found')}")
print(f"  CRM Activities Table: {resources.get('crm_activities_table_name', 'Not found')}")
print(f"  CRM Customers Table: {resources.get('crm_customers_table_name', 'Not found')}")
print(f"  CRM Deals Table: {resources.get('crm_deals_table_name', 'Not found')}")
print(f"  EC2 Role: {resources.get('ec2_role_name', 'Not found')}")
print(f"  Public ALB DNS: {resources.get('public_alb_dns', 'Not found')}")

print("\n✅ Fault injection utilities ready")

## 2. Verify Infrastructure

## 2. 验证基础设施

Before injecting faults, let's verify that the CloudFormation stack has created all necessary resources and they are healthy.

在注入故障之前，让我们验证 CloudFormation 堆栈已创建所有必要的资源并且它们是健康的。

In [ ]:
from lab_helpers.lab_01.infrastructure import (
    verify_ec2_instances,
    verify_dynamodb_tables,
    verify_alb_health,
    verify_cloudwatch_logs
)

print("Verifying infrastructure components...\n")

# Verify EC2 instances are running
ec2_status = verify_ec2_instances(resources, AWS_REGION, AWS_PROFILE)

# Verify DynamoDB tables exist and are accessible
dynamodb_status = verify_dynamodb_tables(resources, AWS_REGION, AWS_PROFILE)

# Verify ALB targets are healthy
alb_status = verify_alb_health(resources, AWS_REGION, AWS_PROFILE)

# Verify CloudWatch log groups exist
logs_status = verify_cloudwatch_logs(AWS_REGION, AWS_PROFILE)

if all([ec2_status, dynamodb_status, alb_status, logs_status]):
    print("\n✅ All infrastructure components verified and healthy")
else:
    print("\n⚠️  Some infrastructure components failed verification")

## 3. Test Fault Injection and Review Pre-Baked Faults

## 3. 测试故障注入并审查预设故障

In this section, we'll inject two infrastructure faults and review two additional faults that are pre-baked into the deployment. Together, these **4 faults** will provide comprehensive training scenarios for your diagnostic agents.

在本节中，我们将注入两个基础设施故障，并审查部署中预设的另外两个故障。这 **4 个故障**将为您的诊断代理提供全面的培训场景。

### Fault 1: DynamoDB Throttling

### 故障 1：DynamoDB 限流

**What it is:**
DynamoDB throttling occurs when your application exceeds the provisioned read/write capacity of your tables. This is a common production issue that can happen when:
- Traffic spikes unexpectedly exceed provisioned capacity
- Tables are misconfigured with insufficient capacity units

**什么是 DynamoDB 限流：**
当您的应用程序超过表的预配置读/写容量时，就会发生 DynamoDB 限流。这是一个常见的生产问题，可能发生在：
- 流量峰值意外超过预配置容量
- 表配置了不足的容量单位

**How we inject this fault:**
The `inject_dynamodb_throttling()` helper function simulates this by:
- Converting the metrics table from `PAY_PER_REQUEST` (unlimited) to `PROVISIONED` billing mode
- Setting extremely low capacity limits: **1 Read Capacity Unit** and **1 Write Capacity Unit**
- This means the table can only handle ~1 read and ~1 write operation per second
- Any normal application load will immediately exceed these limits

**我们如何注入此故障：**
`inject_dynamodb_throttling()` 辅助函数通过以下方式模拟此故障：
- 将指标表从 `PAY_PER_REQUEST`（无限制）转换为 `PROVISIONED` 计费模式
- 设置极低的容量限制：**1 个读取容量单位**和 **1 个写入容量单位**
- 这意味着表每秒只能处理约 1 次读取和约 1 次写入操作
- 任何正常的应用程序负载都会立即超过这些限制

**Expected impact:**
- `ProvisionedThroughputExceededException` errors in application logs
- Increased latency as requests get throttled and retried
- CloudWatch metrics will show throttled requests

**预期影响：**
- 应用程序日志中出现 `ProvisionedThroughputExceededException` 错误
- 由于请求被限流和重试，延迟增加
- CloudWatch 指标将显示被限流的请求

In [ ]:
# Execute DynamoDB throttling fault injection
success = inject_dynamodb_throttling(resources, AWS_REGION, AWS_PROFILE)

if success:
    print("✅ DynamoDB throttling fault injected successfully")
    print("   → Table converted to PROVISIONED mode with 1 RCU/1 WCU")
    print("   → Normal application load will now trigger throttling")
else:
    print("❌ Failed to inject DynamoDB throttling fault")

### Load Application Tables 

### 加载应用程序表

Now lets load test our endpoint. We are going to send 20 concurrent requests/second for 30 seconds. This load is not very significant, but due to misconfiguration in the table capacity provisioned - our app should should show `ProvisionedThroughputExceededException` errors in [application logs](https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsre-workshop$252Fcrm-application) and CloudWatch metrics will show throttled requests. If you try to access the Customers tab during the load test, you will experience issues with it loading the data.

现在让我们对端点进行负载测试。我们将在 30 秒内每秒发送 20 个并发请求。这个负载并不是很大，但由于表容量配置错误 - 我们的应用程序应该在[应用程序日志](https://us-west-2.console.aws.amazon.com/cloudwatch/home?region=us-west-2#logsV2:log-groups/log-group/$252Faws$252Fsre-workshop$252Fcrm-application)中显示 `ProvisionedThroughputExceededException` 错误，CloudWatch 指标将显示被限流的请求。如果您在负载测试期间尝试访问 Customers 选项卡，您将遇到数据加载问题。

In [ ]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor

alb_dns = resources['public_alb_dns']
url = f"http://{alb_dns}:8080/api/customers"

def make_request(i):
    try:
        requests.get(url, timeout=5)
    except:
        pass

for second in range(1, 31):
    with ThreadPoolExecutor(max_workers=50) as executor:
        executor.map(make_request, range(50))

    if second % 10 == 0:
        print(f"Progress: {second}/30 seconds")

    time.sleep(1)

print("\n✓ Load test complete")

### Fault 2: IAM Permission Issues

### 故障 2：IAM 权限问题

**What it is:**
IAM permission issues occur when applications lack necessary permissions to access AWS resources. This is one of the most common production problems, often caused by:
- Overly restrictive security policies applied without testing
- Role assumptions failing due to trust policy modifications
- Security team applying blanket deny policies

**什么是 IAM 权限问题：**
当应用程序缺少访问 AWS 资源所需的权限时，就会发生 IAM 权限问题。这是最常见的生产问题之一，通常由以下原因引起：
- 未经测试就应用了过于严格的安全策略
- 由于信任策略修改导致角色假设失败
- 安全团队应用了全面拒绝策略

**How we inject this fault:**
Our helper function `inject_iam_permissions()` simulates this by:
- Locating the EC2 instance IAM role used by the application servers
- Backing up the original DynamoDB access policy
- Replacing it with an explicit **Deny** policy for key DynamoDB operations
- Targeting: `PutItem`, `GetItem`, `Query`, `Scan`, `UpdateItem`, `DeleteItem`
- Since Deny policies override Allow policies, this immediately blocks database access

**我们如何注入此故障：**
我们的辅助函数 `inject_iam_permissions()` 通过以下方式模拟此故障：
- 定位应用程序服务器使用的 EC2 实例 IAM 角色
- 备份原始 DynamoDB 访问策略
- 将其替换为关键 DynamoDB 操作的显式 **Deny** 策略
- 目标操作：`PutItem`、`GetItem`、`Query`、`Scan`、`UpdateItem`、`DeleteItem`
- 由于 Deny 策略会覆盖 Allow 策略，这会立即阻止数据库访问

**Expected impact:**
- `AccessDenied` exceptions in application logs for any database operations
- Complete failure of features that require DynamoDB access

**预期影响：**
- 任何数据库操作的应用程序日志中出现 `AccessDenied` 异常
- 需要 DynamoDB 访问的功能完全失败

In [ ]:
# Execute IAM permission fault injection
success = inject_iam_permissions(resources, AWS_REGION, AWS_PROFILE)

if success:
    print("✅ IAM permission fault injected successfully")
    print(f"   → EC2 role '{resources.get('ec2_role_name', 'Unknown')}' now has Deny policy")
    print("   → All DynamoDB operations will return AccessDenied")
else:
    print("❌ Failed to inject IAM permission fault")

Let's test what response we get when invoking our application now. We should see error 500 due to the backend issues with the API not being able to retrieve data from DynamoDB.
**Note**: It may take a minute for IAM permissions to propagate. If you're not seeing 500 errors, please wait and try again.

让我们测试一下现在调用应用程序时会得到什么响应。由于 API 无法从 DynamoDB 检索数据的后端问题，我们应该看到 500 错误。
**注意**：IAM 权限可能需要一分钟才能传播。如果您没有看到 500 错误，请等待并重试。

In [ ]:
time.sleep(180)
alb_dns = resources['public_alb_dns']

url = f"http://{alb_dns}:8080/api/deals"

print(f"\nGenerating 5 requests to trigger IAM errors...")
print(f"Target: {url}\n")

for i in range(10):
    try:
        response = requests.get(url, timeout=5)
        print(f"Request {i+1} - Status: {response.status_code}")
    except Exception as e:
        print(f"Request {i+1} - Error: {str(e)}")

    time.sleep(1)  # Small delay to avoid overwhelming

print("\n✓ Load complete - waiting 10 seconds for logs to propagate...")
time.sleep(10)


## Summary

## 总结

✅ Prerequisites verified and infrastructure deployed. CRM application is running and monitored via CloudWatch. We have injected faults simulating real production issues for our Agent to troubleshoot.

✅ 先决条件已验证，基础设施已部署。CRM 应用程序正在运行并通过 CloudWatch 进行监控。我们已注入模拟真实生产问题的故障，供我们的代理进行故障排除。

Next: Lab 2 - Build the Diagnostics Agent (Lab-02-diagnostics-agent.ipynb)

下一步：实验 2 - 构建诊断代理（Lab-02-diagnostics-agent.ipynb）

In [ ]:
# Try url with both port 80 and 8080
print(f"Click here to access the CRM App UI: '{get_app_url()}'")
